In [1]:
# Necessary libraries 
import pandas as pd
import re
import numpy as np
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import Counter
from wordcloud import WordCloud


In [2]:
#Load dataset
df = pd.read_csv("customer_support_tickets.csv")
# Remove duplicates
df.drop_duplicates(inplace=True)
df.dropna(subset=['Ticket Description'], inplace=True)
df.reset_index(drop=True, inplace=True)


In [3]:
# Clean and preprocess
    
# Add custom words    
custom_noise = {'using', 'would', 'soon', 'use', 'purchased','recently', 'facing', 'possible', 
                'assist', 'tried', 'please', 'steps', 'help', 'im', 'need', 'ive', 'persists', 
                'could', 'mentioned', 'unable', 'find', 'thanks', 'support', 'issue', 'us', 'also',
                'regards', 'resolved', 'would', 'problem', 'issue', 'productpurchased', 'product', 
               'noticed', 'made', 'started', 'times', 'customer', 'already', 'multiple', 'used', 'last',
                'seems', 'longer', 'fine', 'available', 'longer', 'followed', 'reviewed', 'checked',
               'sure', 'occurring', 'recently', 'specific', 'changes', 'recent', 'havent', 'devices',
                'option', 'says', 'message', 'screen','might', 'hoping', 'performed', 'properly','original',
                'didnt', 'resolve', 'working', 'thank', 'like', 'similar', 'different', 'others', 'reported',
               'sometimes', 'acts', 'unexpectedly', 'works', 'ensure', 'desired', 'action', 'remains', 
                'unresolved', 'doesnt', 'everything', 'mean', 'peculiar'}

stop_words = set(stopwords.words('english')).union(custom_noise)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
 
    text = (text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    words = text.split()
    cleaned_words = [w for w in words if w not in stop_words and w not in custom_noise]
    
    text = ' ' .join(cleaned_words)

    return text

#Apply Cleaning 
df["Processed Description"] = df["Ticket Description"].apply(preprocess_text)
# Compare text before and after processing
print("\n Original Text vs Processed Text ")
pd.set_option('display.max_colwidth', None)
display(df[['Ticket Description', 'Processed Description']].head())


 Original Text vs Processed Text 


,Ticket Description,Processed Description
0,"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nYour billing zip code is: 71701.\r\n\r\nWe appreciate that you have requested a website address.\r\n\r\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",billing zip code appreciate requested website address double check email address troubleshooting user manual
1,"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf you need to change an existing product.\r\n\r\nI'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",change existing intermittent
2,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\r\n\r\n1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it's not charging properly.",turning yesterday respond really charger came charging
3,"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.",youre interested id love see happen check feedback contacted
4,I'm having an issue with the {product_purchased}. Please assist.\r\n\r\n\r\nNote: The seller is not responsible for any damages arising out of the delivery of the battleground game. Please have the game in good condition and shipped to you I've noticed a sudden decrease in battery life on my {product_purchased}. It used to last much longer.,note seller responsible damages arising delivery battleground game game good condition shipped sudden decrease battery life much


The preprocessing step removed irrelevant words such as 'please' and 'assist', while retaining key terms related to customer issues. 

In [4]:
# Extract cleaned text to list
documents = df["Processed Description"].tolist()
# Remove empty strings
documents = [doc for doc in documents if len(doc.strip()) >0]
# final output
print(f"Documents ready for modelling: {len(documents)} documents.")
print("\nSample of processed documents:")
for doc in documents[:3]:
    print(f"-{doc}")

Documents ready for modelling: 8469 documents.

Sample of processed documents:
-billing zip code appreciate requested website address double check email address troubleshooting user manual
-change existing intermittent
-turning yesterday respond really charger came charging


In [5]:
df.to_csv("cleaned_tickets.csv", index=False)

print("Saved columns:", df.columns)

Saved columns: Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating', 'Processed Description'],
      dtype='object')
